# Stability and the CFL condition

In the previous lesson, we studied the numerical solution of the linear and non-linear convection equations, using the finite-difference method. 
We began by discretizing the one-way wave equation using a classic *forward-time/backward space* scheme, and computing the solution using an initial condition consisting of a square pulse.

Computing with finer spatial grids while keeping the time step constant, you encountered four cases:

1. a coarse run with `nx=41` smoothed the square pulse and attenuated its height,
2. a finer run with `nx=81` improved the solution, but smoothing still ocurred,
3. a surprising case with `nx=101`matched the exact solution, and
4. a further refinement with `nx=121` destroyed the solution.

In this lesson, we will explore why changing the discretization parameters can affect your solution in such a drastic way.
The central question we want to answer is:

> Why did refining the spatial grid improve the linear-convection calculation, make it an exact grid shift at one setting, and then destroy it?

Let's begin by importing our favorite Python libraries for numerical computing.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Set the font family and size to use for Matplotlib figures.
pyplot.rcParams['font.family'] = 'serif'
pyplot.rcParams['font.size'] = 12

The code below corresponds to the same function we used in the previous lesson, but incorporating the vectorized update in space, leaving only the loop in time. Be sure to review the way array slicing works in this code, sketched in Figure 5 of Lesson 6, and the Python refresher that came after.

In [ ]:
def advance_linear_convection(u0, c, dx, dt, num_steps):
    '''Advance linear convection with the FTBS scheme.'''
    u = u0.copy()
    for n in range(num_steps):
        u[1:] = u[1:] - c * dt / dx * (u[1:] - u[:-1])
    return u

### What happened?

To answer that question, we have to think a little bit about what we're actually implementing in the code when we solve the linear convection equation with the forward-time/backward-space method.  

In each iteration of the time loop, we use the existing data about the solution at time $n$ to compute the solution in the subsequent time step, $n+1$.  In the first few cases, the increase in the number of grid points returned more accurate results.  There was less discretization error and the translating wave looked more like a square wave than it did in our first example.  

Each iteration of the time loop advances the solution by a time-step of length $\Delta t$, which had the value 0.025 in the examples above. During this iteration, we evaluate the solution $u$ at each of the $x_i$ points on the grid.  But in the last plot, something has clearly gone wrong.  

What has happened is that over the time period $\Delta t$, the wave is travelling a distance which is greater than `dx`, and we say that the solution becomes *unstable* in this situation (this statement can be proven formally, see below).  The length `dx` of grid spacing is inversely proportional to the number of total points `nx`: we asked for more grid points, so `dx` got smaller. Once `dx` got smaller than the $c\Delta t$—the distance travelled by the numerical solution in one time step—it's no longer possible for the numerical scheme to solve the equation correctly!

![CFLcondition](figures/CFLcondition.png)
#### Graphical interpretation of the CFL condition.

Consider the illustration above. The green triangle represents the _domain of dependence_ of the numerical scheme. Indeed, for each time step, the variable $u_i^{n+1}$ only depends on the values $u_i^{n}$ and $u_{i-1}^{n}$. 

When the distance $c\Delta t$ is smaller than $\Delta x$, the characteristic line traced from the grid coordinate $i, n+1$ lands _between_ the points $i-1,n$ and $i,n$ on the grid. We then say that the _mathematical domain of dependence_ of the solution of the original PDE is contained in the _domain of dependence_ of the numerical scheme. 

On the contrary, if $\Delta x$ is smaller than $c\Delta t$, then the information about the solution needed for $u_i^{n+1}$ is not available in the _domain of dependence_ of the numerical scheme, because the characteristic line traced from the grid coordinate $i, n+1$ lands _behind_ the point $i-1,n$ on the grid. 

The following condition thus ensures that the domain of dependence of the differential equation is contained in the _numerical_ domain of dependence: 

$$
\begin{equation}
\sigma = \frac{c \Delta t}{\Delta x} \leq 1
\end{equation}
$$

As can be proven formally, stability of the numerical solution requires that step size `dt` is calculated with respect to the size of `dx` to satisfy the condition above.  

The value of $c\Delta t/\Delta x$ is called the **Courant-Friedrichs-Lewy number** (CFL number), often denoted by $\sigma$. The value $\sigma_{\text{max}}$ that will ensure stability depends on the discretization used; for the forward-time/backward-space scheme, the condition for stability is $\sigma<1$.

In a new version of our code—written _defensively_—, we'll use the CFL number to calculate the appropriate time-step `dt` depending on the size of `dx`.  
 